## Избор на датасет

Серијата ја мери вкупната дневна побарувачка на електрична енергија во Шпанија.
    Мерењата се во GWh, во периодот од 1 јануари 2019 до 31 декември 2025 (2.557 дневни набљудувања).
    Податоците се преземени од REData API на Red Eléctrica de España (REE), националниот оператор на електроенергетскиот систем. Фреквенцијата е дневна и правилна, без недостасувачки вредности.
Дополнително се користи дневната просечна температура од Open-Meteo Historical API.
    Температурата е пресметана како просек од пет големи градови (Мадрид, Барселона, Валенсија, Севилја, Билбао), додавајки тежини според нивното население.

In [1]:
import requests 
import pandas as pd
from pathlib import Path
import time

In [2]:
RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

In [3]:
URL = "https://apidatos.ree.es/en/datos/demanda/evolucion"
all_values = []

for year in range(2019, 2026):
    params = {
        "start_date": f"{year}-01-01T00:00",
        "end_date": f"{year}-12-31T23:59",
        "time_trunc": "day"
    }
    r = requests.get(URL, params=params, timeout=60)
    r.raise_for_status()
    values = r.json()["included"][0]["attributes"]["values"]
    all_values.extend(values)
    print(year, len(values), "days")
    time.sleep(1)

2019 365 days
2020 366 days
2021 365 days
2022 365 days
2023 365 days
2024 366 days
2025 365 days


In [4]:
df = pd.DataFrame(all_values)
df.to_csv(RAW / "ree_demand_raw.csv", index=False)

print(df.shape)
df.head()

(2557, 3)


,value,percentage,datetime
0,582949.806,1,2019-01-01T00:00:00.000+01:00
1,742199.407,1,2019-01-02T00:00:00.000+01:00
2,787764.963,1,2019-01-03T00:00:00.000+01:00
3,796010.035,1,2019-01-04T00:00:00.000+01:00
4,721029.651,1,2019-01-05T00:00:00.000+01:00


In [5]:
df["date"] = pd.to_datetime(df["datetime"].str[:10])

demand = (df.set_index("date")["value"] / 1000).rename("load_gwh").sort_index()
demand = demand[~demand.index.duplicated()]

demand.head()

date
2019-01-01    582.949806
2019-01-02    742.199407
2019-01-03    787.764963
2019-01-04    796.010035
2019-01-05    721.029651
Name: load_gwh, dtype: float64

In [6]:
print("Rows:",len(demand))
print("From", demand.index.min().date(), "to", demand.index.max().date())

demand.to_csv(PROCESSED / "ree_demand_daily.csv")

Rows: 2557
From 2019-01-01 to 2025-12-31


In [18]:
URL = "https://archive-api.open-meteo.com/v1/archive"
CITIES = {"Madrid":(40.42, -3.70), "Barcelona":(41.38, 2.17), "Valencia":(39.47, -0.38), "Seville":(37.38, -5.98),"Bilbao":(43.26, -2.93)}
temps = {}

for city, (lat, lon) in CITIES.items():
    params = {
        "latitude":lat,
        "longitude":lon,
        "start_date":"2019-01-01",
        "end_date":"2025-12-31",
        "daily":"temperature_2m_mean",
        "timezone":"Europe/Madrid",
    }
    r = requests.get(URL, params=params, timeout=60)
    r.raise_for_status()
    d = r.json()["daily"]
    temps[city] = pd.Series(d["temperature_2m_mean"], index=pd.to_datetime(d["time"]))
    print(city, len(temps[city]), "days")


Madrid 2557 days
Barcelona 2557 days
Valencia 2557 days
Seville 2557 days
Bilbao 2557 days


In [19]:
temp_df = pd.DataFrame(temps)
temp_df.to_csv(RAW / "temperature_raw.csv")

temp_df.head()

,Madrid,Barcelona,Valencia,Seville,Bilbao
2019-01-01,2.9,9.5,9.3,10.0,6.2
2019-01-02,2.6,8.0,9.1,8.5,7.2
2019-01-03,3.6,6.2,9.0,8.3,4.1
2019-01-04,2.8,5.6,8.4,9.7,1.7
2019-01-05,1.9,7.3,10.3,8.9,2.8


In [21]:
WEIGHTS = {"Madrid": 7.0, "Barcelona": 8.0, "Valencia": 5.3,
           "Seville": 8.6, "Bilbao": 2.2}

w = pd.Series(WEIGHTS)
temp = (temp_df[w.index] * w).sum(axis=1) / w.sum()
temp = temp.rename("temp_c")
temp.tail()

2025-12-27     8.865595
2025-12-28    10.233762
2025-12-29     9.373633
2025-12-30     8.198071
2025-12-31     7.209003
Name: temp_c, dtype: float64

In [22]:
data = pd.concat([demand, temp], axis=1).asfreq("D")
print("Missing values:\n", data.isna().sum())

data = data.interpolate(limit=3)
data.to_csv(PROCESSED / "spain_daily.csv")
data.head()

Missing values:
 load_gwh    0
temp_c      0
dtype: int64


,load_gwh,temp_c
2019-01-01,582.949806,7.885209
2019-01-02,742.199407,7.053698
2019-01-03,787.764963,6.524116
2019-01-04,796.010035,6.304823
2019-01-05,721.029651,6.719936
